# Phase 2: Unsupervised Learning for Initial Detection (Anomaly-Based IDS)
# Task 2.1: Packet-Level Anomaly Detection
# Autoencoder

Build an unsupervised model that separates benign from non-benign packets.

## Setup, load metadata and sampled packet dataset

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

METADATA_FILE_NAME = "packet_sample_seed1.metadata.json"
PROJECT_ROOT = Path("..")
META_PATH = PROJECT_ROOT / "data" / "samples" / METADATA_FILE_NAME

with open(META_PATH, "r") as f:
    metadata = json.load(f)

DATA_PATH = PROJECT_ROOT / metadata["output_path"]
df = pd.read_csv(DATA_PATH)

BINARY_COL = "binary_label"
ATTACK_COL = "attack_type"
y_true = df[BINARY_COL].map({"benign": 0, "attack": 1}).to_numpy()

print("Dataset shape:", df.shape)
print("\nBinary class counts:")
print(df[BINARY_COL].value_counts())
print("\nTraffic type counts:")
print(df[ATTACK_COL].value_counts())

Dataset shape: (205041, 138)

Binary class counts:
binary_label
benign    200000
attack      5041
Name: count, dtype: int64

Traffic type counts:
attack_type
Benign             200000
DNS Spoofing         3178
DDoS-HTTP Flood       833
Brute Force           404
DoS-HTTP Flood        364
XSS                   262
Name: count, dtype: int64


## Data preprocessing

### Autoencoder feature selection

The autoencoder uses numeric packet behavior, protocol, timing, size, and short-term traffic statistics. Raw IP addresses, MAC addresses, host names, URIs, and user-agent strings are excluded because they are identifiers rather than stable behavior features. A small set of 1, 5, and 30-window statistics is kept, then highly correlated columns are removed using the training set only.

In [2]:
# Basic packet behavior features
base_features = [
    "time_since_previously_displayed_frame", "port_class_dst", "l4_tcp", "l4_udp", "ttl",
    "eth_size", "tcp_window_size", "payload_entropy", "payload_length", "jitter",
    "dns_len_qry", "dns_interval", "dns_len_ans", "http_content_len", "http_response_code",
    "handshake_cipher_suites_length", "handshake_extensions_length", "handshake_sig_hash_alg_len",
    "icmp_data_size", "l3_ip_dst_count", "average_p", "var_p", "iqr_p"
]

# Keep useful short-term and longer-term traffic statistics
rolling_features = []
for group in ["stream", "src_ip", "channel"]:
    rolling_features += [f"{group}_{window}_count" for window in [1, 5, 30]]
    rolling_features += [f"{group}_{window}_{stat}" for window in [5, 30] for stat in ["mean", "var"]]

rolling_features += ["stream_jitter_5_mean", "stream_jitter_5_var", "stream_jitter_30_mean", "stream_jitter_30_var"]

requested_features = base_features + rolling_features
missing_features = [col for col in requested_features if col not in df.columns]
feature_cols = [col for col in requested_features if col in df.columns]
X_raw = df[feature_cols].apply(pd.to_numeric, errors="coerce").copy()

print("Initial selected features:", X_raw.shape[1])
print("Missing requested features:", missing_features)

Initial selected features: 48
Missing requested features: []


### Autoencoder feature engineering

The added features describe protocol usage, payload presence, port behavior, and sudden traffic bursts. These are more useful than raw port numbers and help the autoencoder learn normal packet relationships without using labels.

In [3]:
src_port = pd.to_numeric(df["src_port"], errors="coerce").fillna(-1)
dst_port = pd.to_numeric(df["dst_port"], errors="coerce").fillna(-1)
packet_size = pd.to_numeric(df["eth_size"], errors="coerce").replace(0, np.nan)
payload_size = pd.to_numeric(df["payload_length"], errors="coerce").fillna(0)
dns_query = pd.to_numeric(df["dns_len_qry"], errors="coerce").fillna(0)
dns_answer = pd.to_numeric(df["dns_len_ans"], errors="coerce").fillna(0)
http_length = pd.to_numeric(df["http_content_len"], errors="coerce").fillna(0)
http_code = pd.to_numeric(df["http_response_code"], errors="coerce").fillna(0)
tls_length = pd.to_numeric(df["handshake_extensions_length"], errors="coerce").fillna(0)
icmp_size = pd.to_numeric(df["icmp_data_size"], errors="coerce").fillna(-1)

X_raw["payload_ratio"] = (payload_size / packet_size).clip(0, 1).fillna(0)
X_raw["has_payload"] = (payload_size > 0).astype(int)
X_raw["uses_dns_port"] = ((src_port == 53) | (dst_port == 53)).astype(int)
X_raw["has_dns_data"] = ((dns_query > 0) | (dns_answer > 0)).astype(int)
X_raw["uses_http_port"] = ((src_port == 80) | (dst_port == 80)).astype(int)
X_raw["uses_https_port"] = ((src_port == 443) | (dst_port == 443)).astype(int)
X_raw["has_http_data"] = ((http_length > 0) | (http_code > 0)).astype(int)
X_raw["has_tls_handshake"] = (tls_length > 0).astype(int)
X_raw["is_icmp"] = (icmp_size >= 0).astype(int)
X_raw["src_port_ephemeral"] = (src_port >= 49152).astype(int)
X_raw["dst_port_ephemeral"] = (dst_port >= 49152).astype(int)

def burst_ratio(short_col, long_col):
    short_count = pd.to_numeric(df[short_col], errors="coerce").fillna(0)
    long_count = pd.to_numeric(df[long_col], errors="coerce").fillna(0)
    recent_rate = short_count / 5
    previous_rate = (long_count - short_count).clip(lower=0) / 25
    return (recent_rate + 1) / (previous_rate + 1)

for group in ["stream", "src_ip", "channel"]:
    X_raw[f"{group}_burst_ratio"] = burst_ratio(f"{group}_5_count", f"{group}_30_count")

print("Features after engineering:", X_raw.shape[1])
X_raw.head()

Features after engineering: 62


,time_since_previously_displayed_frame,port_class_dst,l4_tcp,l4_udp,ttl,eth_size,tcp_window_size,payload_entropy,payload_length,jitter,...,uses_http_port,uses_https_port,has_http_data,has_tls_handshake,is_icmp,src_port_ephemeral,dst_port_ephemeral,stream_burst_ratio,src_ip_burst_ratio,channel_burst_ratio
0,0.000185,1,1,0,64,60,0,0.000000,0,0.000185,...,0,1,0,0,0,0,0,3.800000,1.179039,3.800000
1,0.000290,3,1,0,49,66,65024,0.000000,0,0.000290,...,0,1,0,0,0,0,1,2.800000,1.085714,1.051051
2,0.003855,1,1,0,64,74,14600,0.000000,0,947.623914,...,0,1,0,0,0,1,0,1.200000,1.081081,1.072508
3,0.003104,3,0,1,64,60,0,3.201876,16,0.005255,...,0,0,0,0,0,1,1,2.777778,1.371681,2.826087
4,0.000001,3,1,0,231,66,4415,0.000000,0,0.000001,...,0,1,0,0,0,0,1,1.429706,1.451327,1.429140


### Basic preprocessing

In [4]:
# Create 60% training, 20% validation and 20% test sets
all_idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(all_idx, test_size=0.40, random_state=1, stratify=df[ATTACK_COL])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=1, stratify=df.iloc[temp_idx][ATTACK_COL])

# Clean values before learning preprocessing information
X_clean = X_raw.replace([np.inf, -np.inf], np.nan).copy()
zero_fill_cols = [col for col in X_clean.columns if col.endswith("_var") or "jitter" in col]
X_clean[zero_fill_cols] = X_clean[zero_fill_cols].fillna(0)

X_train_df = X_clean.iloc[train_idx].copy()
X_val_df = X_clean.iloc[val_idx].copy()
X_test_df = X_clean.iloc[test_idx].copy()

# Remove unusable columns using training data only
drop_cols = [col for col in X_train_df.columns if X_train_df[col].isna().all() or X_train_df[col].nunique(dropna=True) <= 1]
X_train_df = X_train_df.drop(columns=drop_cols)
X_val_df = X_val_df.drop(columns=drop_cols)
X_test_df = X_test_df.drop(columns=drop_cols)

# Reduce extreme skew without changing binary indicators
log_cols = []
for col in X_train_df.columns:
    values = X_train_df[col].dropna()
    if len(values) > 0 and values.min() >= 0 and values.nunique() > 2 and abs(values.skew()) > 2:
        log_cols.append(col)

for split in [X_train_df, X_val_df, X_test_df]:
    split[log_cols] = np.log1p(split[log_cols].clip(lower=0))

# Remove nearly duplicate features using training correlations only
CORRELATED_FEATURE_THRESHOLD = 0.98
correlation = X_train_df.corr(numeric_only=True).abs()
upper_triangle = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
correlated_cols = [col for col in upper_triangle.columns if (upper_triangle[col] > CORRELATED_FEATURE_THRESHOLD).any()]

X_train_df = X_train_df.drop(columns=correlated_cols)
X_val_df = X_val_df.drop(columns=correlated_cols)
X_test_df = X_test_df.drop(columns=correlated_cols)

preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

X_train = preprocess.fit_transform(X_train_df).astype(np.float32)
X_val = preprocess.transform(X_val_df).astype(np.float32)
X_test = preprocess.transform(X_test_df).astype(np.float32)
processed_feature_names = preprocess.named_steps["imputer"].get_feature_names_out(X_train_df.columns)

y_train = y_true[train_idx]
y_val = y_true[val_idx]
y_test = y_true[test_idx]

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("\nPreprocessing summary:")
print("Removed unusable features:", len(drop_cols))
print("Removed correlated features:", len(correlated_cols))
print("Log-transformed features:", len(log_cols))
print("Final model inputs:", len(processed_feature_names))
print("All values finite:", np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all())
print("\nAttack counts:")
print("Train:", y_train.sum())
print("Validation:", y_val.sum())
print("Test:", y_test.sum())

Training shape: (123024, 61)
Validation shape: (41008, 61)
Test shape: (41009, 61)

Preprocessing summary:
Removed unusable features: 0
Removed correlated features: 6
Log-transformed features: 45
Final model inputs: 61
All values finite: True

Attack counts:
Train: 3024
Validation: 1008
Test: 1009
